# NB7 — Statistical-features evaluation (traditional models + LOGO)

**Step 3+4 of the plan.** This is a stat-only notebook — NO neural model, NO CAMeL, NO fusion. It
scores the statistical feature sets with the three traditional classifiers and then under
leave-one-generator-out (LOGO), so we can pick **old-5 vs new-5** on the full statistical picture.

Feature sets compared:
- **OLD-5** (proposal lineage, documented): `targeted_ppl, burstiness, ttr, entity_density, discourse_coherence`
- **NEW-5** (feature-expansion result): `burstiness, ttr, quote_ratio, function_word_ratio, compressibility`
- **NEW-6** (new-5 + fallback passive) — reference only; passive was already shown to add ~0 on LOGO

Two evaluations: (A) standard split — fit train, test on test → Accuracy / Precision / Recall /
Macro-F1 / AUC-ROC; (B) LOGO — leave-one-generator-out, test-split held-out generator → per-generator
Macro-F1, mean, worst. Runs top-to-bottom in one Save-Version pass; CPU only, a couple of minutes.

> Reminder from the probe: on LOGO-worst the new-5 beat the old-5 by ~0.5pp (≈1 article, near the
> noise floor). And `quote_ratio` (in new-5) may partly reflect a corpus-construction quote gap
> (human 0.173 vs AI 0.078), so weigh its external validity when deciding.

## 1 · Config

In [1]:
import numpy as np, pandas as pd
P_DATASET = "/kaggle/input/notebooks/bahaaqassem/nb3-build-dataset/dataset.parquet"          # EDIT to your dataset name
P_VSTAT16 = "/kaggle/input/notebooks/bahaaqassem/ph2-nb6e-extract-11-features/vstat16_scaled.parquet"   # EDIT to your dataset name (train-scaled features)

OLD_5 = ["targeted_ppl", "burstiness", "ttr", "entity_density", "discourse_coherence"]
NEW_5 = ["burstiness", "ttr", "quote_ratio", "function_word_ratio", "compressibility"]
NEW_6 = NEW_5 + ["passive_ratio"]                                 # reference (fallback passive)
FEATURE_SETS = {"OLD_5": OLD_5, "NEW_5": NEW_5, "NEW_6": NEW_6}

GENERATORS = ["deepseek", "sonnet", "qwen", "gemini", "gpt", "opus"]
SEED = 42
print("config loaded")

config loaded


## 2 · Load + align

In [2]:
df = pd.read_parquet(P_DATASET)
if "article_id" in df.columns: df = df.set_index("article_id")
v16 = pd.read_parquet(P_VSTAT16)
if "article_id" in v16.columns: v16 = v16.set_index("article_id")

needed = set().union(*FEATURE_SETS.values())
missing = needed - set(v16.columns)
assert not missing, f"vstat16 missing columns: {missing}"
df  = df.loc[df.index.intersection(v16.index)]
v16 = v16.loc[df.index]
assert (df.index == v16.index).all(), "df<->v16 alignment failed"

y   = df["label"].to_numpy()
gen = df["generator"].fillna("__human__").to_numpy()
is_train = (df["split"]=="train").to_numpy()
is_tv    = df["split"].isin(["train","val"]).to_numpy()
is_test  = (df["split"]=="test").to_numpy()
print(f"aligned {len(df)} | label {dict(zip(*np.unique(y, return_counts=True)))} | "
      f"train {is_train.sum()} val {(df['split']=='val').sum()} test {is_test.sum()}")

aligned 7101 | label {np.int64(0): np.int64(3500), np.int64(1): np.int64(3601)} | train 5363 val 645 test 1093


## 3 · Models

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

def make_models():
    return {
        "LogReg": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED),
        "RandomForest": RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=SEED),
        "GradientBoosting": GradientBoostingClassifier(random_state=SEED),  # no class_weight -> sample_weight
    }

def fit_model(name, clf, X, yy):
    if name == "GradientBoosting":
        clf.fit(X, yy, sample_weight=compute_sample_weight("balanced", yy))
    else:
        clf.fit(X, yy)
    return clf
print("models ready")

models ready


## 4 · (A) Standard-split metrics (fit train, evaluate test)

In [4]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
rowsA = []
for fs_name, cols in FEATURE_SETS.items():
    X = v16[cols].to_numpy(np.float32)
    Xtr, ytr = X[is_train], y[is_train]
    Xte, yte = X[is_test],  y[is_test]
    for m_name, clf in make_models().items():
        clf = fit_model(m_name, clf, Xtr, ytr)
        pred = clf.predict(Xte)
        proba = clf.predict_proba(Xte)[:, 1]
        rowsA.append({
            "features": fs_name, "model": m_name, "dim": len(cols),
            "Accuracy": round(100*accuracy_score(yte, pred), 2),
            "Precision": round(100*precision_score(yte, pred, average="macro"), 2),
            "Recall": round(100*recall_score(yte, pred, average="macro"), 2),
            "MacroF1": round(100*f1_score(yte, pred, average="macro"), 2),
            "AUC": round(100*roc_auc_score(yte, proba), 2),
        })
resA = pd.DataFrame(rowsA)
import IPython.display as ipd; print("STANDARD SPLIT:"); ipd.display(resA)

STANDARD SPLIT:


,features,model,dim,Accuracy,Precision,Recall,MacroF1,AUC
0,OLD_5,LogReg,5,80.88,80.87,80.88,80.88,86.98
1,OLD_5,RandomForest,5,83.44,83.46,83.42,83.43,90.45
2,OLD_5,GradientBoosting,5,84.72,84.72,84.72,84.72,90.92
3,NEW_5,LogReg,5,86.09,86.12,86.11,86.09,92.88
4,NEW_5,RandomForest,5,88.75,88.76,88.76,88.75,95.05
5,NEW_5,GradientBoosting,5,88.75,88.80,88.77,88.75,95.32
6,NEW_6,LogReg,6,86.37,86.39,86.39,86.37,93.20
7,NEW_6,RandomForest,6,89.20,89.22,89.22,89.20,95.41
8,NEW_6,GradientBoosting,6,88.75,88.75,88.76,88.75,95.74


## 5 · (B) LOGO — leave-one-generator-out (stat-only)

In [5]:
def logo_stat(cols, m_name):
    # TRAIN = (train+val) humans + (train+val) AI of the other 5 generators
    # TEST  = test-split humans + test-split AI of the held-out generator only
    X = v16[cols].to_numpy(np.float32); human = (y==0); per = {}
    for g in GENERATORS:
        tr = is_tv   & (human | (gen != g))
        te = is_test & (human | (gen == g))
        clf = fit_model(m_name, make_models()[m_name], X[tr], y[tr])
        per[g] = 100.0 * f1_score(y[te], clf.predict(X[te]), average="macro")
    v = np.array(list(per.values())); return per, v.mean(), v.min()

rowsB = []
for fs_name, cols in FEATURE_SETS.items():
    for m_name in make_models():
        per, mean, worst = logo_stat(cols, m_name)
        rowsB.append({"features": fs_name, "model": m_name,
                      "LOGO_mean": round(mean,2), "LOGO_worst": round(worst,2),
                      **{g: round(per[g],1) for g in GENERATORS}})
resB = pd.DataFrame(rowsB)
print("LOGO (leave-one-generator-out):"); ipd.display(resB)

LOGO (leave-one-generator-out):


,features,model,LOGO_mean,LOGO_worst,deepseek,sonnet,qwen,gemini,gpt,opus
0,OLD_5,LogReg,69.84,65.74,70.5,73.3,73.1,70.3,66.1,65.7
1,OLD_5,RandomForest,72.11,68.27,73.1,77.1,73.7,71.8,68.3,68.7
2,OLD_5,GradientBoosting,72.01,66.98,73.0,77.6,71.9,72.9,67.0,69.7
3,NEW_5,LogReg,76.84,70.10,77.2,80.9,79.9,77.7,70.1,75.2
4,NEW_5,RandomForest,80.16,74.37,80.6,85.3,81.0,80.7,74.4,79.1
5,NEW_5,GradientBoosting,79.87,73.86,82.1,84.5,79.9,80.2,73.9,78.7
6,NEW_6,LogReg,77.41,72.09,77.2,81.2,80.2,77.9,72.1,75.8
7,NEW_6,RandomForest,81.12,75.90,82.3,85.1,81.2,81.5,75.9,80.7
8,NEW_6,GradientBoosting,79.38,73.27,82.5,84.1,79.6,78.8,73.3,78.0


## 6 · Old-5 vs New-5 — side by side

In [6]:
def best_row(res, fs, col):
    sub = res[res.features==fs]
    return sub.loc[sub[col].idxmax()]

print("="*60)
print("STANDARD SPLIT (best model per set, by Macro-F1):")
for fs in ["OLD_5","NEW_5"]:
    r = best_row(resA, fs, "MacroF1")
    print(f"  {fs:6}  MacroF1={r.MacroF1}  AUC={r.AUC}  (via {r.model})")
print("\nLOGO (best model per set, by worst fold):")
for fs in ["OLD_5","NEW_5"]:
    r = best_row(resB, fs, "LOGO_worst")
    print(f"  {fs:6}  worst={r.LOGO_worst}  mean={r.LOGO_mean}  (via {r.model})")

o = best_row(resB,"OLD_5","LOGO_worst").LOGO_worst
n = best_row(resB,"NEW_5","LOGO_worst").LOGO_worst
print("="*60)
print(f"LOGO-worst delta (NEW_5 - OLD_5) = {n-o:+.2f}pp   (1 gpt article ~ 0.42pp)")
print("Decision is yours: NEW_5 tends higher on LOGO-worst, but if within ~1 article, OLD_5 is the")
print("documented/proposal-lineage choice. Also weigh quote_ratio's corpus-construction caveat.")

STANDARD SPLIT (best model per set, by Macro-F1):
  OLD_5   MacroF1=84.72  AUC=90.92  (via GradientBoosting)
  NEW_5   MacroF1=88.75  AUC=95.05  (via RandomForest)

LOGO (best model per set, by worst fold):
  OLD_5   worst=68.27  mean=72.11  (via RandomForest)
  NEW_5   worst=74.37  mean=80.16  (via RandomForest)
LOGO-worst delta (NEW_5 - OLD_5) = +6.10pp   (1 gpt article ~ 0.42pp)
Decision is yours: NEW_5 tends higher on LOGO-worst, but if within ~1 article, OLD_5 is the
documented/proposal-lineage choice. Also weigh quote_ratio's corpus-construction caveat.
